# F00 ASSETFORGE — Phase 2 : Image Generation

**Modèle :** FLUX.1-schnell (Apache 2.0, gratuit, HuggingFace)
**Runtime :** GPU T4 (Kaggle)

The manifest is embedded by GitHub Actions as EMBEDDED_MANIFEST.

In [ ]:
# === EMBEDDED MANIFEST (injected by GitHub Actions) ===
# This cell is replaced by the workflow with the actual manifest
EMBEDDED_MANIFEST = None


In [ ]:
# === INSTALL (torch is pre-installed by Kaggle with CUDA) ===
!pip install -q diffusers transformers accelerate
print('Dependencies installed')

In [ ]:
# === IMPORTS ===
import os
import json
import time
import zipfile
from pathlib import Path

import torch
from diffusers import FluxPipeline

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Check Kaggle accelerator settings.')

In [ ]:
# === LOAD MANIFEST ===
if EMBEDDED_MANIFEST is not None:
    manifest = EMBEDDED_MANIFEST
    print(f'Using embedded manifest')
else:
    import glob
    candidates = glob.glob('/kaggle/input/*/prompts_manifest.json')
    if candidates:
        with open(candidates[0]) as f:
            manifest = json.load(f)
        print(f'Found manifest in dataset: {candidates[0]}')
    else:
        raise ValueError('No manifest found — EMBEDDED_MANIFEST is None and no dataset')

meta = manifest['meta']
images = manifest['images']
print(f'Mode: {meta["mode"]}')
print(f'Format: {meta["format"]}')
print(f'Images to generate: {len(images)}')

In [ ]:
# === GPU CHECK + LOAD MODEL ===
if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU NOT AVAILABLE! This notebook requires GPU.\n'
        'On Kaggle: Settings > Accelerator > GPU T4 x2\n'
        f'Current PyTorch: {torch.__version__} (CPU-only)'
    )

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# FLUX.1-schnell — fastest model, ~2-4s/image on T4
pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-schnell',
    torch_dtype=torch.float16
)
pipe = pipe.to('cuda')

# Optimizations for T4 (16GB VRAM)
pipe.enable_attention_slicing()
try:
    pipe.enable_model_cpu_offload()
except Exception:
    pass

print('FLUX.1-schnell loaded on GPU')

In [ ]:
# === GENERATE IMAGES ===
OUTPUT_DIR = Path('/kaggle/working/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if meta['format'] == 'VERTICAL':
    WIDTH, HEIGHT = 1080, 1920
else:
    WIDTH, HEIGHT = 1920, 1080

NUM_STEPS = 4  # FLUX.1-schnell works in 4 steps
GUIDANCE = 0.0  # schnell doesn't use guidance scale

generated = []
failed = []

for i, img_spec in enumerate(images):
    filename = img_spec['filename']
    prompt = img_spec['prompt']
    
    style_constraints = (
        f'No text, no watermark, no logo in the image. '
        f'High contrast, clear composition. '
        f'Consistent visual style across all images.'
    )
    full_prompt = f'{prompt}. {style_constraints}'
    
    print(f'[{i+1}/{len(images)}] Generating {filename}...')
    t0 = time.time()
    
    try:
        result = pipe(
            prompt=full_prompt,
            width=WIDTH,
            height=HEIGHT,
            num_inference_steps=NUM_STEPS,
            guidance_scale=GUIDANCE,
        )
        image = result.images[0]
        image.save(OUTPUT_DIR / filename)
        elapsed = time.time() - t0
        print(f'  OK {filename} ({elapsed:.1f}s)')
        generated.append(filename)
    except Exception as e:
        print(f'  FAILED: {e}')
        failed.append({'filename': filename, 'error': str(e)})
    
    torch.cuda.empty_cache()

print(f'\nGenerated: {len(generated)}/{len(images)}')
if failed:
    print(f'Failed: {len(failed)}')
    for f in failed:
        print(f'  - {f["filename"]}: {f["error"][:80]}')

In [ ]:
# === ZIP OUTPUT ===
ZIP_PATH = '/kaggle/working/f00_images.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for img_file in sorted(OUTPUT_DIR.iterdir()):
        if img_file.is_file():
            zf.write(img_file, img_file.name)

print(f'Output zipped: {ZIP_PATH}')
print(f'Files in zip: {len(generated)}')
print(f'Size: {os.path.getsize(ZIP_PATH) / 1024 / 1024:.1f} MB')

In [ ]:
# === SAVE METADATA ===
metadata = {
    'model': 'FLUX.1-schnell',
    'format': meta['format'],
    'mode': meta['mode'],
    'total_requested': len(images),
    'total_generated': len(generated),
    'failed': failed,
    'generation_params': {
        'steps': NUM_STEPS,
        'guidance': GUIDANCE,
        'width': WIDTH,
        'height': HEIGHT,
    },
}

with open('/kaggle/working/generation_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Metadata saved.')
print(f'\nF00 ASSETFORGE Phase 2 — COMPLETE')
print(f'Images: {len(generated)}/{len(images)}')